# Step 6: Property Value Interpolation & EIF Census Enrichment

Russell Blessing

## Overview

This notebook does two related things:

**Property value interpolation.** For parcels matched to a mitigation or application record but carrying an unreliable `TOTAL.VALUE.CALCULATED` (vacant land, government-owned, anomalously low values), estimate a market-comparable value by averaging the recorded values of the nearest 1–4 family residential donor parcels. Three estimates per recipient (k = 5, 10, 50 nearest neighbors).

**EIF census enrichment.** For each mit/app point, attach Census Bureau Gridded EIF attributes from three release years (1999, 2020, 2024) at the 0.01° grid cell containing that point. Seven EIF files are joined:

-   Housing units × householder age × income × home value — 2024 only (no historical version exists; column prefix `hu_2024_`)
-   Population × age × race × sex — 1999, 2020, 2024 (prefixes `ars_1999_`, `ars_2020_`, `ars_2024_`)
-   Population × race × income decile — 1999, 2020, 2024 (prefixes `ri_1999_`, `ri_2020_`, `ri_2024_`)

The three years cover the bookends and a midpoint of the project’s 1999–2024 disaster history. Year-matched demographics let downstream analysis pull contextual demographics from the same year as each mit/app’s program record (e.g., a 1999 Hurricane Floyd buyout record can use `ars_1999_*` columns rather than 2024 baseline demographics).

Both stages consume the **alt Step 5 resolved outputs** as the single source of mit/app data. Each record carries an `assigned_parcel_index` (resolved to one parcel per record, no fan-out) and a point geometry. The interpolation uses `assigned_parcel_index` to count mits/apps per parcel; the EIF enrichment uses the point geometry to snap each record to its grid cell.

**Inputs:**

-   `parcels_pri.gpkg` — parcels with priority scoring (canonical Step 5)
-   `alt_mits_resolved.gpkg` — point-geometry mits, one row per record, resolved to one assigned parcel each (alt Step 5)
-   `alt_apps_resolved.gpkg` — point-geometry apps, same structure (alt Step 5)
-   Seven Gridded EIF parquet files (downloaded on first run)

**Outputs (written to `out_dir`):**

-   `interpolate_to.gpkg`, `interpolate_from.gpkg` — recipient and donor pools
-   `interpolated_vals.gpkg`, `interpolated_parcels_final.csv` — parcels with interpolated values
-   `mits_with_eif.gpkg`, `apps_with_eif.gpkg` — alt resolved outputs with EIF cell attributes
-   `mits_with_eif.csv`, `apps_with_eif.csv` — same, no geometry

## Methodology

**Recipients (parcels with unreliable recorded property values)**: parcels that have at least one mit or app assigned to them by alt Step 5 AND whose recorded `TOTAL.VALUE.CALCULATED` is treated as unreliable. A value is treated as unreliable if the parcel is:

-   vacant land (`vacant == 1`) — no structure, recorded value reflects land only
-   government land use (`gov_lu == 1`) — typically tax-exempt with \$0 assessed
-   government-owned by owner name (`gov_owned == 1`) — same
-   low recorded value (`TOTAL.VALUE.CALCULATED < $25,000`) — anomalously low, likely indicates undervaluation, recent transfer, or data quality issue

**Donors (parcels providing value estimates)**: 1–4 family residential, NOT government-owned, with non-missing recorded total value above \$25,000. Strict NA exclusion ensures every neighbor mean is computed over exactly k actual values.

**Mit/app per-parcel counts** are computed from `assigned_parcel_index` in the alt resolved outputs — each unique mit/app contributes to exactly one parcel. This avoids the fan-out double-counting that would happen if we counted from canonical `mits_pcls.gpkg` (where a mit hitting N parcels appears in N rows). Records with `assignment_method == "unassigned"` are excluded from counts.

**Government-ownership detection** uses a regex on `OWNER.1.FULL.NAME` matching common government-entity keywords (`CITY`, `STATE`, `TOWN`, `VILLAGE`, `COUNTY`, `METROPOLITAN`, plus abbreviations) excluding `INC` and `LLC`.

**EIF cell join** uses cell-membership rather than spatial intersection — each mit/app point’s lat/lon is snapped to its 0.01° grid cell centroid via `floor(coord * 100) / 100 + 0.005` (cell centers are at `.005` offsets per the EIF grid topology), then joined to EIF data by the snapped coordinates as a character key. Equivalent result, much faster than `st_join`.

**Stage 2 nearest-neighbor search** uses parcel centroid distance rather than point-to-polygon-edge distance for computational reliability. For NC residential parcels (~30–80m typical diameter), centroid distance overstates edge distance by a similar amount; the 200m distance cap still correctly identifies geocoding-artifact rescues. Earlier polygon-distance implementation via `nngeo::st_nn` produced misaligned neighbor results and was replaced.

**Caveats** worth noting in the manuscript:

-   The \$25,000 thresholds are heuristic; sensitivity analysis at \$10k and \$50k recommended.
-   Three K values (5/10/50) are produced in parallel; downstream analysis picks the granularity.
-   EIF counts are differentially-private — approximate per cell.
-   Adjacent buildings within ~1.2 km² of each other share the same EIF cell. For dense urban geometry this means per-cell attributes can’t distinguish neighboring properties.
-   Donor pool is statewide; border parcels may reach across counties.

In [ ]:
library(sf)


Linking to GEOS 3.12.0, GDAL 3.11.0, PROJ 9.2.1; sf_use_s2() is TRUE


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Attaching package: 'arrow'

The following object is masked from 'package:utils':

    timestamp

In [ ]:
out_dir         <- "/proj/mhinolab/users/rbless/data/Obstacles_Output"
parcels_path    <- file.path(out_dir, "parcels_pri.gpkg")
mits_alt_path   <- file.path(out_dir, "alt_mits_resolved.gpkg")
apps_alt_path   <- file.path(out_dir, "alt_apps_resolved.gpkg")

# Tunables — interpolation (unchanged)
LOW_VALUE_THRESHOLD <- 25000
MIN_DONOR_VALUE     <- 25000
K_VALUES            <- c(5, 10, 50)

# Tunables — government ownership regex (unchanged)
GOV_KEYWORDS     <- "\\bCITY\\b|\\bSTATE\\b|\\bTOWN\\b|\\bVILLAGE\\b|\\bCOUNTY\\b|\\bMETROPOLITAN\\b|\\bCTY\\b|\\bST\\b|\\bTWN\\b|\\bVILL\\b|\\bVIL\\b|\\bCNTY\\b|\\bMETRO\\b"
EXCLUDE_KEYWORDS <- "\\bINC\\b|\\bLLC\\b"

# EIF source files — multi-year for ageracesex and raceincome,
# 2024-only for hu_age_homeval (no historical version exists for that file).
EIF_BASE_URL  <- "https://www2.census.gov/ces/gridded_eif"
EIF_POP_YEARS <- c(1999, 2020)

build_eif_files <- function() {
  files <- list()

  # Population × age × race × sex — multiple years
  for (yr in EIF_POP_YEARS) {
    name <- paste0("ageracesex_", yr)
    files[[name]] <- list(
      url    = file.path(EIF_BASE_URL, sprintf("gridded_eif_pop_ageracesex_%d.parquet", yr)),
      path   = file.path(out_dir,      sprintf("gridded_eif_pop_ageracesex_%d.parquet", yr)),
      prefix = paste0("ars_", yr)
    )
  }

  # Population × race × income decile — multiple years
  for (yr in EIF_POP_YEARS) {
    name <- paste0("raceincome_", yr)
    files[[name]] <- list(
      url    = file.path(EIF_BASE_URL, sprintf("gridded_eif_pop_raceincome_%d.parquet", yr)),
      path   = file.path(out_dir,      sprintf("gridded_eif_pop_raceincome_%d.parquet", yr)),
      prefix = paste0("ri_", yr)
    )
  }

  files
}

eif_files <- build_eif_files()

# NC bounding box (WGS84) with small buffer for cells crossing the boundary
NC_BBOX <- list(lat_min = 33.7, lat_max = 36.7,
                lon_min = -84.5, lon_max = -75.4)


## 1 Load parcels and alt mits/apps

In [ ]:
parcels <- st_read(parcels_path, quiet = TRUE) |>
  st_transform(32119)

cat("Parcels loaded:", nrow(parcels), "rows\n")


Parcels loaded: 4216239 rows

In [ ]:
mits_resolved <- st_read(mits_alt_path, quiet = TRUE)
apps_resolved <- st_read(apps_alt_path, quiet = TRUE)

cat("Mits (alt resolved):", nrow(mits_resolved), "rows\n")


Mits (alt resolved): 8801 rows

Apps (alt resolved): 23232 rows


Mit assignment_method distribution:

      assignment_method    n
1  direct_single_parcel 7113
2     nearest_high_addr  935
3     nearest_high_dist  689
4            unassigned   40
5 direct_multi_resolved   24


App assignment_method distribution:

      assignment_method     n
1  direct_single_parcel 20792
2     nearest_high_dist  1214
3     nearest_high_addr  1015
4            unassigned   150
5 direct_multi_resolved    61

## 2 Aggregate mit/app counts per parcel

One mit/app contributes to exactly one parcel — counted by `assigned_parcel_index` from the alt resolved outputs. Records with `assignment_method == "unassigned"` (no parcel within reach) are excluded. Parcels with no matched mits/apps get a zero count.

In [ ]:
mit_counts <- mits_resolved |>
  st_drop_geometry() |>
  filter(!is.na(assigned_parcel_index)) |>
  count(assigned_parcel_index, name = "mit_count") |>
  rename(parcel_index = assigned_parcel_index)

app_counts <- apps_resolved |>
  st_drop_geometry() |>
  filter(!is.na(assigned_parcel_index)) |>
  count(assigned_parcel_index, name = "app_count") |>
  rename(parcel_index = assigned_parcel_index)

parcels <- parcels |>
  left_join(mit_counts, by = "parcel_index") |>
  left_join(app_counts, by = "parcel_index") |>
  mutate(
    mit_count = replace_na(mit_count, 0L),
    app_count = replace_na(app_count, 0L)
  )

cat("Parcels with at least one assigned mit:", sum(parcels$mit_count > 0), "\n")


Parcels with at least one assigned mit: 6524 

Parcels with at least one assigned app: 9146 

Parcels with both:                      4527 

## 3 Identify government-owned parcels

In [ ]:
parcels <- parcels |>
  mutate(
    owner_name_clean = replace_na(`OWNER.1.FULL.NAME`, ""),
    gov_owned = as.integer(
      str_detect(owner_name_clean, regex(GOV_KEYWORDS,    ignore_case = TRUE)) &
      !str_detect(owner_name_clean, regex(EXCLUDE_KEYWORDS, ignore_case = TRUE))
    )
  ) |>
  select(-owner_name_clean)

cat("Government-owned parcels (by owner name):", sum(parcels$gov_owned), "\n")


Government-owned parcels (by owner name): 58673 

Overlap with gov_lu (by land use code):   20050 

## 4 Type conversions

`TOTAL.VALUE.CALCULATED` and `IMPROVEMENT.VALUE.CALCULATED` came in as character. Coerce to numeric for the threshold filters and KNN means.

In [ ]:
parcels <- parcels |>
  mutate(
    `TOTAL.VALUE.CALCULATED`       = suppressWarnings(as.numeric(`TOTAL.VALUE.CALCULATED`)),
    `IMPROVEMENT.VALUE.CALCULATED` = suppressWarnings(as.numeric(`IMPROVEMENT.VALUE.CALCULATED`)),
    fr_1_4 = as.integer(fr_1_4)
  )


## 5 Define recipient and donor pools

In [ ]:
to_interpolate <- parcels |>
  filter(
    (mit_count > 0 | app_count > 0),
    (
      vacant == 1 |
      gov_lu == 1 |
      gov_owned == 1 |
      (!is.na(`TOTAL.VALUE.CALCULATED`) & `TOTAL.VALUE.CALCULATED` < LOW_VALUE_THRESHOLD)
    )
  )

interpolate_from <- parcels |>
  filter(
    fr_1_4 == 1,
    gov_owned == 0,
    !is.na(`TOTAL.VALUE.CALCULATED`),
    `TOTAL.VALUE.CALCULATED` > MIN_DONOR_VALUE
  )

cat("Recipients (to_interpolate):", nrow(to_interpolate), "\n")


Recipients (to_interpolate): 5456 

Donors (interpolate_from):   2833343 

## 6 K-nearest-neighbor interpolation

For each recipient parcel, find the K nearest donor parcels by centroid distance, then average their `TOTAL.VALUE.CALCULATED`. Three values of K (5, 10, 50) computed in parallel.

In [ ]:
from_coords <- st_coordinates(st_centroid(interpolate_from))


Recipient row count: 5456 

All prop_value_5 non-NA: TRUE 

All prop_value_10 non-NA: TRUE 

All prop_value_50 non-NA: TRUE 

## 7 Join interpolated values back to all parcels

In [ ]:
to_interp_attrs <- to_interpolate |>
  st_drop_geometry() |>
  select(parcel_index, prop_value_5, prop_value_10, prop_value_50)

joined <- parcels |>
  left_join(to_interp_attrs, by = "parcel_index") |>
  mutate(
    interpolate = as.integer(parcel_index %in% to_interpolate$parcel_index),
    prop_value_5  = coalesce(prop_value_5,  `TOTAL.VALUE.CALCULATED`),
    prop_value_10 = coalesce(prop_value_10, `TOTAL.VALUE.CALCULATED`),
    prop_value_50 = coalesce(prop_value_50, `TOTAL.VALUE.CALCULATED`)
  )

cat("Total parcels in joined output:", nrow(joined), "\n")


Total parcels in joined output: 4216239 

Parcels marked as interpolated: 5456 

## 8 Write interpolated parcels output

In [ ]:
st_write(joined, file.path(out_dir, "interpolated_vals.gpkg"),
         delete_dsn = TRUE, quiet = TRUE)

joined |>
  st_drop_geometry() |>
  write_csv(file.path(out_dir, "interpolated_parcels_final.csv"))


## 9 Gridded EIF Census enrichment

The Census Bureau Gridded EIF provides privacy-protected counts on a fixed 0.01° grid (~1.2 km² per cell). Seven files across three years are joined to each mit/app point:

| File | Cross-tab | Prefix | Years |
|------------------|------------------|------------------|------------------|
| `gridded_eif_hu_age_homeval_2024.parquet` | housing units × householder age × income × home value | `hu_2024_` | 2024 only |
| `gridded_eif_pop_ageracesex_{year}.parquet` | population × age × race × sex | `ars_{year}_` | 1999, 2020, 2024 |
| `gridded_eif_pop_raceincome_{year}.parquet` | population × race × income decile | `ri_{year}_` | 1999, 2020, 2024 |

Cell-membership join via `floor(coord * 100) / 100 + 0.005` — equivalent to spatial intersection but faster.

### 9.1 Download EIF files (one-time)

In [ ]:
for (name in names(eif_files)) {
  f <- eif_files[[name]]
  if (!file.exists(f$path)) {
    message("Downloading ", name, " -> ", basename(f$path))
    tryCatch(
      download.file(f$url, f$path, mode = "wb"),
      error = function(e) {
        stop("EIF download failed for ", name, ": ", conditionMessage(e),
             "\nIf HPC egress is restricted, download manually via wget ",
             "from a login node and place at ", f$path)
      }
    )
  } else {
    message("Already present: ", basename(f$path))
  }
}


Already present: gridded_eif_pop_ageracesex_1999.parquet

Already present: gridded_eif_pop_ageracesex_2020.parquet

Already present: gridded_eif_pop_raceincome_1999.parquet

Already present: gridded_eif_pop_raceincome_2020.parquet

### 9.2 Inspect EIF schemas

In [ ]:
for (name in names(eif_files)) {
  f <- eif_files[[name]]
  cat("\n=== ", name, " ===\n")
  print(arrow::open_dataset(f$path)$schema)
}



===  ageracesex_1999  ===
Schema
grid_lon: string
grid_lat: string
age_group: string
race_ethnicity: string
sex: string
n_noise: double
n_noise_postprocessed: double

===  ageracesex_2020  ===
Schema
grid_lon: string
grid_lat: string
age_group: string
race_ethnicity: string
sex: string
n_noise: double
n_noise_postprocessed: double

===  raceincome_1999  ===
Schema
grid_lon: string
grid_lat: string
income_decile: double
race_ethnicity: string
n_noise: double
n_noise_postprocessed: double

===  raceincome_2020  ===
Schema
grid_lon: string
grid_lat: string
income_decile: double
race_ethnicity: string
n_noise: double
n_noise_postprocessed: double

### 9.3 Read EIF parquets (filtered to NC bounding box)

In [ ]:
LAT_COL <- "grid_lat"
LON_COL <- "grid_lon"

read_eif <- function(path, prefix) {
  d <- arrow::open_dataset(path) |>
    collect()

  d <- d |>
    mutate(
      grid_lat_num = as.numeric(.data[[LAT_COL]]),
      grid_lon_num = as.numeric(.data[[LON_COL]])
    ) |>
    filter(
      grid_lat_num >= NC_BBOX$lat_min,
      grid_lat_num <= NC_BBOX$lat_max,
      grid_lon_num >= NC_BBOX$lon_min,
      grid_lon_num <= NC_BBOX$lon_max
    ) |>
    mutate(
      eif_lat = round(grid_lat_num, 3),
      eif_lon = round(grid_lon_num, 3)
    ) |>
    select(-grid_lat_num, -grid_lon_num, -any_of(c(LAT_COL, LON_COL)))

  # Identify demographic grouping columns (everything except cell key and counts)
  count_cols <- c("n_noise", "n_noise_postprocessed")
  group_cols <- setdiff(names(d), c("eif_lat", "eif_lon", count_cols))

  # Pivot to wide: one row per cell, columns named {prefix}_{group}_{count}
  if (length(group_cols) > 0) {
    d <- d |>
      unite("_group_key", all_of(group_cols), sep = "__", remove = TRUE) |>
      pivot_wider(
        id_cols     = c("eif_lat", "eif_lon"),
        names_from  = "_group_key",
        values_from = all_of(count_cols),
        names_glue  = paste0(prefix, "_{.value}__{.name}")
      )
  } else {
    d <- d |>
      rename_with(~ paste0(prefix, "_", .x), all_of(count_cols))
  }

  d
}

eif_data <- setNames(
  map(eif_files, function(f) read_eif(f$path, f$prefix)),
  names(eif_files)
)

cat("EIF cells in NC bounding box (by dataset):\n")


EIF cells in NC bounding box (by dataset):

  ageracesex_1999                146,468 rows, 128 columns
  ageracesex_2020                165,061 rows, 118 columns
  raceincome_1999                146,468 rows, 134 columns
  raceincome_2020                165,061 rows, 134 columns

### 9.4 Snap mit/app points to EIF cells

In [ ]:
add_eif_cell <- function(sf_obj) {
  coords <- sf_obj |> st_transform(4326) |> st_coordinates()
  sf_obj |>
    mutate(
      eif_lon = round(floor(coords[, "X"] * 100) / 100 + 0.005, 3),
      eif_lat = round(floor(coords[, "Y"] * 100) / 100 + 0.005, 3)
    )
}

mits_resolved <- add_eif_cell(mits_resolved)
apps_resolved <- add_eif_cell(apps_resolved)


### 9.5 Join all EIF datasets onto mits/apps

In [ ]:
join_eif_all <- function(sf_obj) {
  result <- sf_obj
  for (nm in names(eif_data)) {
    result <- left_join(result, eif_data[[nm]], by = c("eif_lat", "eif_lon"))
  }
  result
}

mits_with_eif <- join_eif_all(mits_resolved)
apps_with_eif <- join_eif_all(apps_resolved)

# Coverage diagnostics
coverage <- function(df, label) {
  cat("\n", label, "EIF coverage (cells matched / total records):\n")
  for (nm in names(eif_files)) {
    pfx <- eif_files[[nm]]$prefix
    marker <- names(df)[startsWith(names(df), paste0(pfx, "_"))][1]
    if (!is.na(marker)) {
      n <- sum(!is.na(df[[marker]]))
      cat(sprintf("  %-30s %d / %d (%.1f%%)\n", nm, n, nrow(df), 100 * n / nrow(df)))
    }
  }
}

coverage(mits_with_eif, "Mitigations")



 Mitigations EIF coverage (cells matched / total records):
  ageracesex_1999                7018 / 8801 (79.7%)
  ageracesex_2020                7990 / 8801 (90.8%)
  raceincome_1999                6838 / 8801 (77.7%)
  raceincome_2020                685 / 8801 (7.8%)


 Applications EIF coverage (cells matched / total records):
  ageracesex_1999                15245 / 23232 (65.6%)
  ageracesex_2020                21224 / 23232 (91.4%)
  raceincome_1999                17290 / 23232 (74.4%)
  raceincome_2020                1981 / 23232 (8.5%)

### 9.6 Write enriched outputs

In [ ]:
st_write(mits_with_eif, file.path(out_dir, "mits_with_eif.gpkg"),
         delete_dsn = TRUE, quiet = TRUE)
st_write(apps_with_eif, file.path(out_dir, "apps_with_eif.gpkg"),
         delete_dsn = TRUE, quiet = TRUE)

mits_with_eif |> st_drop_geometry() |>
  write_csv(file.path(out_dir, "mits_with_eif.csv"))
apps_with_eif |> st_drop_geometry() |>
  write_csv(file.path(out_dir, "apps_with_eif.csv"))


## Summary

In [ ]:
hu_marker <- paste0("hu_2024_", LAT_COL)

eif_coverage <- sapply(names(eif_files), function(nm) {
  pfx    <- eif_files[[nm]]$prefix
  marker <- names(mits_with_eif)[startsWith(names(mits_with_eif), paste0(pfx, "_"))][1]
  if (!is.na(marker)) sum(!is.na(mits_with_eif[[marker]])) else NA_integer_
})

summary_tbl <- tibble(
  metric = c(
    "Total parcels",
    "Recipients (interpolated)",
    "Donors (used for averaging)",
    "Median original TOTAL.VALUE.CALCULATED (recipients only)",
    "Median prop_value_5",
    "Median prop_value_10",
    "Median prop_value_50",
    paste0("Mits with EIF cell match — ", names(eif_files))
  ),
  value = c(
    nrow(joined),
    sum(joined$interpolate == 1),
    nrow(interpolate_from),
    median(to_interpolate$`TOTAL.VALUE.CALCULATED`, na.rm = TRUE),
    median(to_interpolate$prop_value_5,  na.rm = TRUE),
    median(to_interpolate$prop_value_10, na.rm = TRUE),
    median(to_interpolate$prop_value_50, na.rm = TRUE),
    eif_coverage
  )
)

knitr::kable(summary_tbl, format.args = list(big.mark = ","))


  metric                                                             value
  ---------------------------------------------------------- -------------
  Total parcels                                                4,216,239.0
  Recipients (interpolated)                                        5,456.0
  Donors (used for averaging)                                  2,833,343.0
  Median original TOTAL.VALUE.CALCULATED (recipients only)         8,500.0
  Median prop_value_5                                             64,324.6
  Median prop_value_10                                            69,036.3
  Median prop_value_50                                            78,450.0
  Mits with EIF cell match --- ageracesex_1999                     7,018.0
  Mits with EIF cell match --- ageracesex_2020                     7,990.0
  Mits with EIF cell match --- raceincome_1999                     6,838.0
  Mits with EIF cell match --- raceincome_2020                       685.0


In [ ]:
to_interpolate |>
  st_drop_geometry() |>
  select(parcel_index, original = `TOTAL.VALUE.CALCULATED`,
         prop_value_5, prop_value_10, prop_value_50) |>
  pivot_longer(-parcel_index, names_to = "metric", values_to = "value") |>
  group_by(metric) |>
  summarise(
    n      = sum(!is.na(value)),
    p10    = quantile(value, 0.10, na.rm = TRUE),
    median = median(value,         na.rm = TRUE),
    mean   = mean(value,           na.rm = TRUE),
    p90    = quantile(value, 0.90, na.rm = TRUE),
    .groups = "drop"
  ) |>
  knitr::kable(
    format.args = list(big.mark = ",", scientific = FALSE),
    caption = "Distribution of original vs. interpolated values for recipient parcels"
  )


  metric                n         p10     median         mean         p90
  --------------- ------- ----------- ---------- ------------ -----------
  original          5,446    1,500.00    8,500.0   299,817.30    41,745.0
  prop_value_10     5,456   38,967.25   69,036.3   103,999.81   176,157.7
  prop_value_5      5,456   38,507.50   64,324.6    97,962.25   168,859.1
  prop_value_50     5,456   45,409.30   78,450.0   102,299.25   161,201.5

  : Distribution of original vs. interpolated values for recipient
  parcels


## Output schema

**`interpolated_vals.gpkg`** / **`interpolated_parcels_final.csv`** — one row per parcel:

| Column | Source | Notes |
|------------------------|------------------------|------------------------|
| (all original parcel columns) | parcels_pri.gpkg | unchanged |
| `mit_count` | derived | per-parcel count of mits assigned via alt Step 5 |
| `app_count` | derived | per-parcel count of apps assigned via alt Step 5 |
| `gov_owned` | derived | 1 if owner name matches gov regex (excluding INC/LLC) |
| `interpolate` | derived | 1 if parcel was a recipient (had value estimated) |
| `prop_value_5` | derived | mean of 5 nearest donors; original if not interpolated |
| `prop_value_10` | derived | mean of 10 nearest donors; original if not interpolated |
| `prop_value_50` | derived | mean of 50 nearest donors; original if not interpolated |

**`mits_with_eif.gpkg`** / **`apps_with_eif.gpkg`** — one row per mit/app:

| Column | Source | Notes |
|------------------------|------------------------|------------------------|
| (all original alt-resolved columns) | alt Step 5 outputs | unchanged |
| `eif_lat`, `eif_lon` | derived | snapped 0.01° cell-center key |
| `hu_2024_*` columns | EIF hu_age_homeval 2024 | housing units, householder age, home value |
| `ars_1999_*`, `ars_2020_*`, `ars_2024_*` columns | EIF pop_ageracesex (3 years) | population × age × race × sex |
| `ri_1999_*`, `ri_2020_*`, `ri_2024_*` columns | EIF pop_raceincome (3 years) | population × race × income decile |

## Unresolved — confirm before using outputs

-   **`LOW_VALUE_THRESHOLD = 25000`** and **`MIN_DONOR_VALUE = 25000`** are heuristic. Sensitivity analysis at \$10k and \$50k recommended.
-   **`K_VALUES = c(5, 10, 50)`** — three parallel estimates. Drop unused K values once a preferred granularity is selected.
-   **No within-county constraint** on donor matching. Border parcels may pull donors across county lines.
-   **Government-ownership regex** is empirically derived. Spot-check `gov_owned == 1` against `parusedesc` to confirm coverage.
-   **EIF `LAT_COL` / `LON_COL`** are set to `"grid_lat"` / `"grid_lon"` per the Gridded EIF topology file (CES-WP-24-74). Verified via the `eif-inspect-schemas` chunk at runtime.
-   **EIF differential privacy** — per-cell counts are noise-protected. Negligible for averages across many cells, important for per-cell point estimates.
-   **`unassigned` mits/apps** are excluded from per-parcel counts but still appear in the EIF-enriched outputs (with NA `assigned_parcel_index`). Downstream analysis should decide whether to include or exclude these records depending on whether the analytic unit is the mit/app record or the matched parcel.